## Data processing

Builds `data/processed/publication_data.csv`, the single country-year table behind `01_data_insights.ipynb`. Four datasets from three sources are standardised to ISO-3 country codes and merged:

| source | contents |
|---|---|
| NSF Science & Engineering Indicators<br>1. all S&E publications - SPBS-2<br>2. psychology S&E publications - SPBS-15 | aggregate publication counts, fractional |
| World Bank World Development Indicators | 16 economic and education indicators |
| Taiwan DGBAS | population and GDP per capita |

Taiwan is not a World Bank reporter, so its population and GDP per capita come from DGBAS instead and are marked in `data_source_flag`.

### The 16 World Bank indicators

Pulled broadly here, then narrowed in analysis. The six marked * are the standard measures of wealth, education, and research capacity that `01_data_insights.ipynb` in the correlation analysis with psychology's share.

| role | indicators |
|---|---|
| income |- *GDP per capita (current US$)<br>- GDP per capita, PPP (current and constant 2021 international $)<br>- GNI per capita (Atlas, PPP current, PPP constant 2021) |
| education | - *school enrollment, tertiary (% gross)<br> - government expenditure on education (% of GDP; % of government expenditure)<br> - expenditure on tertiary education (% of education expenditure) |
| research | - *R&D expenditure (% of GDP)<br> - *researchers in R&D (per million people)<br> - scientific and technical journal articles|
| infrastructure | - *individuals using the internet (% of population)<br> - *urban population (% of total) |
| demographics | - population, total |

The six income series are near-collinear, so the models use GDP per capita (current US$) and hold the others as alternates. 

Coverage varies widely. Researchers per million and R&D expenditure are missing for more than half of country-years, checked in the coverage section below. Series codes and definitions: [`data/README.md`](../data/README.md).

In [1]:
%load_ext autoreload
%autoreload 2

import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from iso_codes import name_to_iso3, iso3_to_name, load_nsf_table

# openpyxl warns about the stylesheet in both NSF workbooks; the data reads fine
warnings.filterwarnings("ignore", category=UserWarning,
                        module="openpyxl.styles.stylesheet")


### Country codes

ISO-3 codes are the merge key across all four sources. NSF, the World Bank, and pycountry each spell a handful of countries differently, so the pycountry lookups in `iso_codes.py` are paired with explicit override maps (`nsf_iso_map`, `country_name_fixes`).


## NSF Science & Engineering (S&E) publications

Both NSF tables share a layout (country rows, year columns, and regional aggregate rows), so one loader handles both: `load_nsf_table()` in `iso_codes.py` drops the aggregate rows, reshapes to country-year, and standardizes to ISO-3, printing any labels it could not match.

In [2]:
# All S&E publications
se_all_df = load_nsf_table(
    "../data/raw/nsf/SE_all_2003-2022.xlsx",
    sheet_name="Table SPBS-2",
    value_name="SE_articles_total",
    drop_cols=["Income level"],
)

# Psychology S&E publications
psych_df = load_nsf_table(
    "../data/raw/nsf/SE_psych_2003-2022.xlsx",
    sheet_name="Table SPBS-15",
    value_name="Publications",
)

total_pubs = psych_df["Publications"].sum()
recent_pubs = psych_df.loc[
    psych_df["Year"].between(2020, 2022),
    "Publications"
].sum()

print(f"\nNSF psychology S&E countries retained: {psych_df['Country'].nunique()}")
print(f"NSF total S&E countries retained: {se_all_df['Country'].nunique()}")
print(f"NSF psychology S&E years included: {psych_df['Year'].min()}-{psych_df['Year'].max()}")
print(f"NSF total S&E countries years included: {se_all_df['Year'].min()}-{se_all_df['Year'].max()}")

print(f"\nTotal psychology publications, 2003-2022: {total_pubs:,.0f}")
print(f"Psychology publications, 2020-2022 subset: {recent_pubs:,.0f} ({recent_pubs / total_pubs:.1%} of all psychology publications)")

Table SPBS-2: unmatched ISO codes = 0
Table SPBS-2: unmatched country names = 0


Table SPBS-15: unmatched ISO codes = 0
Table SPBS-15: unmatched country names = 0

NSF psychology S&E countries retained: 201
NSF total S&E countries retained: 201
NSF psychology S&E years included: 2003-2022
NSF total S&E countries years included: 2003-2022

Total psychology publications, 2003-2022: 815,406
Psychology publications, 2020-2022 subset: 187,411 (23.0% of all psychology publications)


In [3]:
nsf_df = psych_df.merge(
    se_all_df[["iso_alpha", "Year", "SE_articles_total"]],
    on=["iso_alpha", "Year"],
    how="left",
)

missing_denom = nsf_df["SE_articles_total"].isna().sum()
print(f"NSF country-years missing a total S&E denominator: {missing_denom} of {len(nsf_df)}")
if missing_denom:
    display(nsf_df.loc[nsf_df["SE_articles_total"].isna(), ["Country", "Year", "Publications"]])

NSF country-years missing a total S&E denominator: 0 of 4020


## World Bank development indicators


In [4]:
wb_data = pd.read_csv('../data/raw/wbi/35cba9f8-915e-41cf-95b9-1ccc17410135_Data.csv')

print(f"World Bank extract: {len(wb_data):,} rows, "
      f"{wb_data['Series Name'].nunique()} indicators, "
      f"{wb_data['Country Name'].nunique()} countries")

World Bank extract: 4,245 rows, 16 indicators, 267 countries


In [5]:
wb_year_cols = [
    col for col in wb_data.columns
    if '[YR' in col
]

for col in wb_year_cols:
    wb_data[col] = (
        wb_data[col]
        .replace('..', np.nan)
        .astype(float)
    )

wb_df = wb_data.melt(
    id_vars=[
        'Country Name',
        'Country Code',
        'Series Name'
    ],
    value_vars=wb_year_cols,
    var_name='Year',
    value_name='Value'
)

wb_df['Year'] = (
    wb_df['Year']
    .str.extract(r'(\d{4})')
    .astype(int)
)

wb_df['Value'] = pd.to_numeric(
    wb_df['Value'],
    errors='coerce'
)

wb_df = (
    wb_df
    .pivot_table(
        index=[
            'Country Name',
            'Country Code',
            'Year'
        ],
        columns='Series Name',
        values='Value',
        aggfunc='first'
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

print(f"Reshaped to {len(wb_df):,} country-years x {wb_df.shape[1] - 3} indicators, "
      f"{wb_df['Year'].min()}-{wb_df['Year'].max()}")

Reshaped to 5,280 country-years x 16 indicators, 2003-2022


## Taiwan (DGBAS)

Taiwan is not a UN member and so is absent from the World Bank tables, but it is a top-15 science producer. Population and GDP per capita come from its national statistics office instead.


In [6]:
taiwan_pop = (
    pd.read_csv(
        "../data/raw/dgbas/E018101010_015528288.csv",
        skiprows=2,
        encoding="big5",
    )
    .rename(columns={
        "Period": "Year",
        "Population (Mid-Year,Persons)": "Population",
        "Per Capita GDP ( U.S.$,at Current Prices )": "GDP_per_capita",
    })
)

# Keep rows with a 4 digit year
taiwan_pop = taiwan_pop[
    taiwan_pop["Year"].astype(str).str.fullmatch(r"\d{4}")
].copy()
taiwan_pop["Year"] = taiwan_pop["Year"].astype(int)

print(f"Taiwan supplemental data: {taiwan_pop['Year'].min()}-{taiwan_pop['Year'].max()}")

Taiwan supplemental data: 2003-2022


## Merge


In [7]:
merged_data = nsf_df.merge(
    wb_df,
    left_on=['iso_alpha','Year'],
    right_on=['Country Code','Year'],
    how='left'
)

merged_data = merged_data.rename(columns={
    'GDP per capita (current US$)': 'GDP_per_capita',
    'GDP per capita, PPP (current international $)': 'GDP_per_capita_PPP',
    'GDP per capita, PPP (constant 2021 international $)': 'GDP_per_capita_PPP_constant',
    'GNI per capita, PPP (current international $)': 'GNI_per_capita_PPP',
    'GNI per capita, PPP (constant 2021 international $)': 'GNI_per_capita_PPP_constant',
    'GNI per capita, Atlas method (current US$)': 'GNI_per_capita_Atlas',
    'Population, total': 'Population',
    'School enrollment, tertiary (% gross)': 'Tertiary_enrollment_pct',
    'Research and development expenditure (% of GDP)': 'RnD_expenditure_pct',
    'Researchers in R&D (per million people)': 'Researchers_per_million',
    'Government expenditure on education, total (% of GDP)': 'Education_expenditure_pct',
    'Government expenditure on education, total (% of government expenditure)': 'Education_expenditure_pct_of_total',
    'Expenditure on tertiary education (% of government expenditure on education)': 'Tertiary_education_expenditure_pct',
    'Individuals using the Internet (% of population)': 'Internet_users_pct',
    'Urban population (% of total population)': 'Urban_population_pct',
    'Scientific and technical journal articles': 'Sci_tech_articles',
})

# rows whose year falls inside the World Bank extract
overlap = merged_data['Year'].isin(wb_df['Year'].unique())

print(f"Merged {len(merged_data):,} country-years, "
      f"{overlap.sum():,} within World Bank year coverage")

Merged 4,020 country-years, 4,020 within World Bank year coverage


Flagged Taiwan rows with `data_source_flag`.


In [8]:
# DGBAS stands in for the World Bank columns Taiwan has no entry for
taiwan_mask = merged_data["Country"] == "Taiwan"
assert taiwan_mask.any(), "Taiwan missing from merged_data['Country']"

taiwan_by_year = taiwan_pop.set_index("Year")
taiwan_years = merged_data.loc[taiwan_mask, "Year"]
for col in ["Population", "GDP_per_capita"]:
    merged_data.loc[taiwan_mask, col] = taiwan_years.map(taiwan_by_year[col])

merged_data["data_source_flag"] = "WB/UNESCO"
merged_data.loc[taiwan_mask, "data_source_flag"] = "DGBAS Taiwan (supplemental; non-WB coverage)"

print(f"Taiwan country-years patched from DGBAS: {taiwan_mask.sum()}")

Taiwan country-years patched from DGBAS: 20


### Coverage

Coverage by WB indicator.


In [9]:
unmatched_wbi_countries = (merged_data[overlap & merged_data['Country Code'].isna()]
                           .groupby(['Country', 'iso_alpha'], as_index=False)['Publications'].sum())

print(f"Countries with no World Bank match: {len(unmatched_wbi_countries)}")
if len(unmatched_wbi_countries):
    print(unmatched_wbi_countries.to_string(index=False))

# keep only the standardized Country/iso_alpha columns carried over from NSF
merged_data = merged_data.drop(columns=['Country Name', 'Country Code'])

Countries with no World Bank match: 4
     Country iso_alpha  Publications
Cook Islands       COK          0.22
        Niue       NIU          0.00
      Taiwan       TWN       5063.58
Vatican City       VAT          4.50


In [10]:
# Remove duplicate columns if present after merge
coverage_data = merged_data.loc[overlap, ~merged_data.columns.duplicated()]

indicator_cols = [
    col for col in coverage_data.columns
    if col not in ['iso_alpha', 'Country', 'Publications', 'Year', 'data_source_flag']
]

print(f"\nCoverage per indicator ({coverage_data['Year'].min()}-{coverage_data['Year'].max()}, "
      f"{len(coverage_data)} country-years):")
for col in indicator_cols:
    non_null = coverage_data[col].notna().sum()
    pct = non_null / len(coverage_data) * 100
    flag = " *low*" if pct < 50 else ""
    print(f"{col}: {non_null}/{len(coverage_data)} ({pct:.1f}%){flag}")


Coverage per indicator (2003-2022, 4020 country-years):
SE_articles_total: 4020/4020 (100.0%)
Tertiary_education_expenditure_pct: 1288/4020 (32.0%) *low*
GDP_per_capita: 3906/4020 (97.2%)
GDP_per_capita_PPP_constant: 3765/4020 (93.7%)
GDP_per_capita_PPP: 3788/4020 (94.2%)
GNI_per_capita_Atlas: 3785/4020 (94.2%)
GNI_per_capita_PPP_constant: 2879/4020 (71.6%)
GNI_per_capita_PPP: 3735/4020 (92.9%)
Education_expenditure_pct: 2733/4020 (68.0%)
Education_expenditure_pct_of_total: 2631/4020 (65.4%)
Internet_users_pct: 3754/4020 (93.4%)
Population: 3960/4020 (98.5%)
RnD_expenditure_pct: 1784/4020 (44.4%) *low*
Researchers_per_million: 1475/4020 (36.7%) *low*
Tertiary_enrollment_pct: 2533/4020 (63.0%)
Sci_tech_articles: 3940/4020 (98.0%)
Urban_population_pct: 3940/4020 (98.0%)


In [11]:
print(f"""
Final merged dataset:
Years included: {merged_data['Year'].min()}-{merged_data['Year'].max()}
Country-year rows: {len(merged_data):,}
Distinct countries: {merged_data['Country'].nunique()}
Psychology publications: {merged_data['Publications'].sum():,.0f}
S&E denominator present: {merged_data['SE_articles_total'].notna().mean():.1%} of rows
""")


Final merged dataset:
Years included: 2003-2022
Country-year rows: 4,020
Distinct countries: 201
Psychology publications: 815,406
S&E denominator present: 100.0% of rows



### Export

Written to `data/processed/publication_data.csv` for `01_data_insights.ipynb`.


In [12]:
merged_data.to_csv("../data/processed/publication_data.csv", index=False)